# Practice run of analysing/testing different models on the UNSW_NB15 dataset, before trying Deep Learning.

Prior research suggests this is a largely non-linear, less separable dataset so deep learning may be necessary, but I will try simpler, more interpretable models first for the sake of completeness, and to gain Variable Importances

In [1]:
#import packages:


from google.colab import drive

try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

if IN_COLAB:
  # Check if drive is mounted by looking for the mount point in the file system.
  # This is a more robust approach than relying on potentially internal variables.
  import os
  if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
  else:
    print("Google Drive is already mounted.")
else:
  print("Not running in Google Colab. Drive mounting skipped.")

from IPython import get_ipython
from IPython.display import display
import cudf
import cuml
from cuml.preprocessing import StandardScaler
from cuml.model_selection import StratifiedKFold, GridSearchCV
from cuml.linear_model import LogisticRegression
from cuml.pipeline import Pipeline
from cuml.tree import DecisionTreeClassifier
from cuml.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm


print("cuML version:", cuml.__version__)
print("New run: Packages loaded")

Google Drive is already mounted.


ModuleNotFoundError: No module named 'cuml.tree'

Let's load our packages and data

In [ ]:
#if using colabs - will need to first mount your drive

#change these for different users
test_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_testing-set.parquet'
training_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_training-set.parquet'

# Import the two CSV files
test_set = pd.read_parquet(test_set_filepath)
train_set = pd.read_parquet(training_set_filepath)

print("Data loaded")


The next cell does some basic analysis, and one hot encodes some of the features:

In [ ]:
def preprocess_data(data_set):
    # Remove 'attack_cat' column if it exists
    if 'attack_cat' in data_set.columns.tolist():
        data_set.drop('attack_cat', axis=1, inplace=True)

    if 'proto' in data_set.columns.tolist():
        # Ensure 'proto' is of type 'object' to avoid categorical issues
        data_set['proto'] = data_set['proto'].astype(str)

        # Calculate percentage occurrences of each category
        category_percentages = data_set['proto'].value_counts(normalize=True) * 100
        top_6_categories = category_percentages.head(6).index.tolist()

        # Group less frequent categories under 'other' using vectorized operations
        data_set['proto_grouped'] = data_set['proto']
        data_set.loc[~data_set['proto'].isin(top_6_categories), 'proto_grouped'] = 'other'

        # Drop the original 'proto' column
        data_set.drop('proto', axis=1, inplace=True)

        # One-hot encode the 'proto_grouped' column
        data_set = pd.get_dummies(data_set, columns=['proto_grouped'], prefix='proto_grouped')

    # One-hot encode any remaining categorical columns
    categorical_cols = data_set.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        data_set = pd.get_dummies(data_set, columns=categorical_cols, prefix_sep='_')

    # Convert boolean columns to integers
    binary_cols = data_set.select_dtypes(include=['bool']).columns

    if not binary_cols.empty:
        data_set[binary_cols] = data_set[binary_cols].astype(int)

    return data_set


# Preprocess the datasets
train_set = preprocess_data(train_set)
test_set = preprocess_data(test_set)

# Identify missing columns
train_columns = set(train_set.columns)
test_columns = set(test_set.columns)

missing_in_test = train_columns - test_columns
missing_in_train = test_columns - train_columns

# Remove 'label' from missing columns if present
missing_in_test.discard('label')
missing_in_train.discard('label')

# Add missing columns to test_set
for col in missing_in_test:
    test_set[col] = 0

# Add missing columns to train_set
for col in missing_in_train:
    train_set[col] = 0

# Ensure the columns are in the same order
common_columns = sorted(train_set.columns)

train_set = train_set[common_columns]
test_set = test_set[common_columns]

# Verify the columns
print(f"Number of columns in train_set: {len(train_set.columns)}")
print(f"Number of columns in test_set: {len(test_set.columns)}")

print(f"Columns in train_set: {train_set.columns.tolist()}")
print(f"Columns in test_set: {test_set.columns.tolist()}")

#turn both pandas dataframes into cudf
train_set = cudf.DataFrame.from_pandas(train_set)
test_set = cudf.DataFrame.from_pandas(test_set)

print("Data preprocessed")

NOTE TO SELF -
1. THIS IS FOR BINARY CLASSIFICATION, WE WANT MULTICLASS EVENTUALLY, BUT FOR NOW WE WILL JUST DO BN


Based on the high number of columns in the Proto column, we may want to consider an Embeddings layer with the Deep Learning that we plan to undertake later. However since DT/RF perform somewhat poorly on sparse vector datasets (like one hot encoded ones) we will group all the extremely rare categories into an 'other'.


In [ ]:
import cudf
import cuml
from cuml.preprocessing import StandardScaler
from cuml.linear_model import LogisticRegression
from cuml.model_selection import StratifiedKFold, GridSearchCV
from tqdm import tqdm
import numpy as np

def run_models(model_type, test_set=test_set, train_set=train_set):
    """
    Runs Logistic Regression (LR), Decision Tree (DT), or Random Forest (RF) model on the dataframe.
    """

    # Ensure data is in cuDF format
    if not isinstance(train_set, cudf.DataFrame):
        train_set = cudf.DataFrame.from_pandas(train_set)
    if not isinstance(test_set, cudf.DataFrame):
        test_set = cudf.DataFrame.from_pandas(test_set)

    # Drop 'label' column and define targets
    X_train = train_set.drop('label', axis=1)
    y_train = train_set['label']
    X_test = test_set.drop('label', axis=1)
    y_test = test_set['label']

    # Convert target variables to cuDF Series if not already
    if not isinstance(y_train, cudf.Series):
        y_train = cudf.Series(y_train)
    if not isinstance(y_test, cudf.Series):
        y_test = cudf.Series(y_test)

    # Scale the data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #=============================== Logistic Regression (LR) ========================================#
    if model_type.upper() == 'LR':
        # Initialize the Logistic Regression model
        model = LogisticRegression(max_iter=1000)

        # Define the hyperparameter grid
        param_grid = {
            'C': [0.01, 0.1, 1, 10, 100],
            # Note: cuML's LogisticRegression supports 'l2' penalty only
        }

        # Set up cross-validation strategies using cuML's StratifiedKFold
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        print("KFolds defined, beginning nested cross-validation.")

        # Outer loop (on training data)
        outer_scores = []

        # Convert y_train to NumPy array for indexing
        y_train_np = y_train.to_numpy()

        for train_index, val_index in tqdm(outer_cv.split(X_train_scaled, y_train_np)):
            # Split the data into training and validation folds
            X_train_fold = X_train_scaled.iloc[train_index]
            y_train_fold = y_train.iloc[train_index]
            X_val_fold = X_train_scaled.iloc[val_index]
            y_val_fold = y_train.iloc[val_index]

            # Set up GridSearchCV with the model (inner loop)
            grid_search = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                cv=inner_cv,
                scoring='roc_auc',
            )

            # Fit GridSearchCV on the training fold
            grid_search.fit(X_train_fold, y_train_fold)

            # Evaluate on the validation fold
            score = grid_search.score(X_val_fold, y_val_fold)
            outer_scores.append(score)

        # Print the average outer score (ROC AUC) on validation data
        print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

        # Evaluate the best model on the held-out test set
        best_model = grid_search.best_estimator_
        test_score = best_model.score(X_test_scaled, y_test)
        print(f"Test ROC AUC: {test_score}")

    #=============================== Decision Tree (DT) ========================================#
    elif model_type.upper() == 'DT':

        # Initialize the Decision Tree model
        model = DecisionTreeClassifier()

        # Define the hyperparameter grid
        param_grid_dt = {
            'max_depth': [3, 5, 10],
            'min_samples_split': [2, 10, 20],
            'min_samples_leaf': [1, 5, 10],
        }

        # Set up cross-validation strategies
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        print("KFolds defined, beginning nested cross-validation for Decision Tree.")

        # Outer loop
        outer_scores = []
        y_train_np = y_train.to_numpy()

        for train_index, val_index in tqdm(outer_cv.split(X_train_scaled, y_train_np)):
            X_train_fold = X_train_scaled.iloc[train_index]
            y_train_fold = y_train.iloc[train_index]
            X_val_fold = X_train_scaled.iloc[val_index]
            y_val_fold = y_train.iloc[val_index]

            # Set up GridSearchCV with the model
            grid_search = GridSearchCV(
                estimator=model,
                param_grid=param_grid_dt,
                cv=inner_cv,
                scoring='roc_auc',
            )

            # Fit GridSearchCV on the training fold
            grid_search.fit(X_train_fold, y_train_fold)

            # Evaluate on the validation fold
            score = grid_search.score(X_val_fold, y_val_fold)
            outer_scores.append(score)

        print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

        # Evaluate the best model on the test set
        best_model = grid_search.best_estimator_
        test_score = best_model.score(X_test_scaled, y_test)
        print(f"Test ROC AUC on Test Set: {test_score}")

    #=============================== Random Forest (RF) ========================================#
    elif model_type.upper() == 'RF':
        from cuml.ensemble import RandomForestClassifier

        # Initialize the Random Forest model
        model = RandomForestClassifier()

        # Define the hyperparameter grid
        param_grid_rf = {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 10],
            'min_samples_split': [2, 10, 20],
            'min_samples_leaf': [1, 5, 10],
        }

        # Set up cross-validation strategies
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        print("KFolds defined, beginning nested cross-validation for Random Forest.")

        # Outer loop
        outer_scores = []
        y_train_np = y_train.to_numpy()

        for train_index, val_index in tqdm(outer_cv.split(X_train_scaled, y_train_np)):
            X_train_fold = X_train_scaled.iloc[train_index]
            y_train_fold = y_train.iloc[train_index]
            X_val_fold = X_train_scaled.iloc[val_index]
            y_val_fold = y_train.iloc[val_index]

            # Set up GridSearchCV with the model
            grid_search = GridSearchCV(
                estimator=model,
                param_grid=param_grid_rf,
                cv=inner_cv,
                scoring='roc_auc',
            )

            # Fit GridSearchCV on the training fold
            grid_search.fit(X_train_fold, y_train_fold)

            # Evaluate on the validation fold
            score = grid_search.score(X_val_fold, y_val_fold)
            outer_scores.append(score)

        print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

        # Evaluate the best model on the test set
        best_model = grid_search.best_estimator_
        test_score = best_model.score(X_test_scaled, y_test)
        print(f"Test ROC AUC on Test Set: {test_score}")

    else:
        print("Invalid model type. Please choose 'LR' for Logistic Regression, 'DT' for Decision Tree, or 'RF' for Random Forest.")

# Example usage:
run_models('LR')  # For Logistic Regression
run_models('DT')  # For Decision Tree
run_models('RF')  # For Random Forest

run_models('LR')